In [6]:
import sys
sys.path.insert(0, '/app')
%load_ext autoreload
%autoreload 2
%reload_ext autoreload
    
import pytest
import pandas as pd
from pathlib import Path
from typing import List, Optional

from features.base_dataframe import BaseDataFrame
from features.features_utils import FeatureType
from merger.merger_utils import merge_multiple_dataframes_from_parquet
from core.data_org import (
    BUNDLE_DIR,
    get_normalised_instrument_dir,
    MktDataTFreq,
    ExchangeNAME,
    ProductType,
)
from core.enums import g_index_col
from norm.norm_utils import load_normalized_df
from strat.strat_backtest import ETF_LIST

etf_list = ['VOO',
'VEA',
'VWO',
'IEF',
'LQD',
'VNQ',
'GLD',
'DBC',
'SCHD',
'QQQ']

etf_short_list = ['QQQ', 'SPY', 'TLT', 'GLD', 'VWO']
etf_short_list_product_type = [
    ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF
]

asset_list = ['VOO',
'VEA',
'VWO',
'IEF',
'LQD',
'VNQ',
'GLD',
'DBC',
'SCHD',
'QQQ',
'EURUSD',
'USDJPY',
'GBPUSD',
'AUDUSD',
'USDCAD',
'USDCHF',
'NZDUSD',
'EURGBP',
'USDCNH',
'USDHKD']
asset_product_type = [
    ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,
    ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,ProductType.ETF,
    ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,
    ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT,ProductType.SPOT
]
fx_list = [
'EURUSD',
'USDJPY',
'GBPUSD',
'AUDUSD',
'USDCAD',
'USDCHF',
'NZDUSD',
'EURGBP',
'USDCNH',
'USDHKD']




DATA_FREQ = MktDataTFreq.CANDLE_1HOUR
SOURCE = ExchangeNAME.FIRSTRATE
PRODUCT_TYPE = ProductType.ETF
ASSET_LIST = asset_list

def get_base_df(freq = DATA_FREQ, source = SOURCE, product_type= PRODUCT_TYPE, symbol = 'QQQ'):

    instrument_dir = get_normalised_instrument_dir(freq, source, product_type, symbol)

    parquet_files = list(instrument_dir.glob("*.df.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {instrument_dir}")

    input_file = parquet_files[0]
    print(f"Loading: {input_file}")

    df = load_normalized_df(str(input_file))
    base_df = BaseDataFrame(
        p_df=df,
        p_valid_col_name="valid_row",
        p_scaling=-1,
        p_verbose=False,
    )
    return base_df

QQQ_base_df = get_base_df(symbol = 'QQQ')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/QQQ/QQQ_*_*_candle_1hour.df.parquet


In [4]:
assets_base_dfs = []
for sym in fx_list:
    assets_base_dfs.append(get_base_df(product_type=ProductType.SPOT, symbol = sym))

for sym in etf_list:
    assets_base_dfs.append(get_base_df(product_type=ProductType.ETF, symbol = sym))


assets_short_list_dfs = []
for sym in etf_short_list:
    assets_short_list_dfs.append(get_base_df(product_type=ProductType.ETF, symbol = sym))



Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/EURUSD/EURUSD_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/USDJPY/USDJPY_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/GBPUSD/GBPUSD_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/AUDUSD/AUDUSD_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/USDCAD/USDCAD_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/USDCHF/USDCHF_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/NZDUSD/NZDUSD_20100103_20260313_candle_1hour.df.parquet
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/spot/EURGBP/EURGBP_20100103_20260313_candle_1hour.df.parquet
Loading:

In [4]:
"""
Unit tests for ETF feature bundle computation.

Tests building features for ETFs and merging them into a bundle using the fwk modules.
"""


DATA_FREQ = MktDataTFreq.CANDLE_1HOUR
SOURCE = ExchangeNAME.FIRSTRATE
PRODUCT_TYPE = ProductType.ETF

FEATURE_TYPES = [
    FeatureType.PRICE,
    FeatureType.RETURN,
    FeatureType.LOG_RETURN,
    FeatureType.VOLUME,
    FeatureType.HIST_VOLATILITY,
    FeatureType.LAG_DELTAS,
    FeatureType.RSI,
    FeatureType.EMA,
    FeatureType.SPREAD_REL_EMA,
    FeatureType.DIFF_REL_EMA_MID,
    FeatureType.ADI,
    FeatureType.ROC,
]

In [5]:
def compute_features_for_asset(freq:MktDataTFreq = DATA_FREQ, source:ExchangeNAME = SOURCE, product_type:ProductType = PRODUCT_TYPE , symbol: str = 'QQQ', features : bool = False) -> Path:
    """
    Compute features for a single Asset and save to bundle directory.

    Returns:
        Path to the saved feature parquet file
    """
    instrument_dir = get_normalised_instrument_dir(freq, source, product_type, symbol)

    parquet_files = list(instrument_dir.glob("*.df.parquet"))
    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in {instrument_dir}")

    input_file = parquet_files[0]
    print(f"Loading: {input_file}")

    df = load_normalized_df(str(input_file))
    print(f"  Loaded {df.shape[0]} rows, {df.shape[1]} columns")

    base_df = BaseDataFrame(
        p_df=df,
        p_valid_col_name="valid_row",
        p_scaling=-1,
        p_verbose=False,
    )

    if features:
        for feature_type in FEATURE_TYPES:
            base_df.add_feature(feature_type)

    base_df.convert_f16_columns()

    result_df = base_df.get_dataframe()
    feature_cols = base_df.get_feature_columns()

    print(f"  Result: {result_df.shape[0]} rows, {len(feature_cols)} features")

    output_dir = BUNDLE_DIR / DATA_FREQ.value
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / f"{symbol}_features.parquet"

    result_df.to_parquet(output_file, index=False)
    print(f"  Saved: {output_file}")

    return output_file


def compute_asset_bundle(
    asset_product_type: Optional[List[str]] = None,
    asset_list: Optional[List[str]] = None,
    output_prefix: str = "asset_bundle",
    output_dir: Path = BUNDLE_DIR,
    features: bool = False
) -> Path:
   
    if asset_product_type is None:
        return null
    if asset_list is None:
        return null
    if output_dir is None:
        output_dir = BUNDLE_DIR

    output_dir.mkdir(parents=True, exist_ok=True)

    feature_files = []
    for i in range(len(asset_list)):
        print(f"\n{'=' * 60}")
        print(f"Processing: {asset_list[i]}")
        print(f"{'=' * 60}")
        feature_file = compute_features_for_asset(product_type = asset_product_type[i],symbol = asset_list[i], features = features)
        feature_files.append(feature_file)

    print(f"\n{'=' * 60}")
    print("Merging All feature files...")
    print(f"{'=' * 60}")

    merged_df = merge_multiple_dataframes_from_parquet(
        file_paths=[str(f) for f in feature_files],
        p_names=asset_list,
        p_cols_list=[],
        p_id_col=g_index_col,
        p_float_16=False,
    )

    print(f"\nMerged dataframe:")
    print(f"  Shape: {merged_df.shape}")

    feature_cols = [c for c in merged_df.columns if "_F_" in c]
    print(f"  Total feature columns: {len(feature_cols)}")

    output_path = output_dir / f"{output_prefix}_features_bundle.parquet"
    merged_df.to_parquet(output_path)

    file_size_mb = output_path.stat().st_size / (1024 * 1024)
    print(f"\n{'=' * 60}")
    print(f"Bundle saved to: {output_path}")
    print(f"File size: {file_size_mb:.2f} MB")
    print(f"{'=' * 60}")

    return output_path

In [7]:
output_file = compute_asset_bundle(
    asset_product_type = etf_short_list_product_type,
    asset_list = etf_short_list ,
    output_prefix="etf_short_list",
    output_dir=BUNDLE_DIR,
)

print(f"\n{'=' * 60}")
print(f"Bundle file: {output_file}")
print(f"{'=' * 60}")

assert output_file.exists(), f"Bundle file should exist: {output_file}"


Processing: QQQ
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/QQQ/QQQ_*_*_candle_1hour.df.parquet
  Loaded 92274 rows, 8 columns
  Result: 92274 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/QQQ_features.parquet

Processing: SPY
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/SPY/SPY_*_*_candle_1hour.df.parquet
  Loaded 96352 rows, 8 columns
  Result: 96352 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/SPY_features.parquet

Processing: TLT
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/TLT/TLT_*_*_candle_1hour.df.parquet
  Loaded 69956 rows, 8 columns
  Result: 69956 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/TLT_features.parquet

Processing: GLD
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/GLD/GLD_*_*_candle_1hour.df.parquet
  Loaded 77307 rows, 8 columns
  Result: 77307 rows, 4 features
  Saved: /app/data/bundle/candle_1hour/GLD_features.parquet

Processing: VWO
Loading

In [ ]:
def test_compute_features_for_single_asset():
    """
    Test computing features for a single ETF (QQQ).
    """
    output_file = compute_features_for_asset(symbol = "QQQ")

    print(f"\n{'=' * 60}")
    print(f"Output file: {output_file}")
    print(f"{'=' * 60}")

    assert output_file.exists(), f"Output file should exist: {output_file}"

    df = pd.read_parquet(output_file)
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()[:10]}...")

    assert df.shape[0] > 0, "DataFrame should have rows"

    feature_cols = [c for c in df.columns if c.startswith("F_")]
    assert len(feature_cols) > 0, "Should have feature columns"

    print(f"\n{'=' * 60}")
    print("TEST PASSED: Single ETF feature computation")
    print(f"{'=' * 60}")

In [15]:
def test_compute_asset_bundle_small():
    """
    """
    output_file = compute_asset_bundle(
        asset_product_type = asset_product_type,
        asset_list = asset_list ,
        output_prefix="test_assets",
        output_dir=BUNDLE_DIR,
    )

    print(f"\n{'=' * 60}")
    print(f"Bundle file: {output_file}")
    print(f"{'=' * 60}")

    assert output_file.exists(), f"Bundle file should exist: {output_file}"

    df = pd.read_parquet(output_file)
    print(f"Shape: {df.shape}")
    print(f"Index: {df.index.name}")

    feature_cols = [c for c in df.columns if "_F_" in c]
    print(f"Feature columns: {len(feature_cols)}")

    assert df.shape[0] > 0, "DataFrame should have rows"
    assert len(feature_cols) > 0, "Should have feature columns"

    for asset in asset_list:
        asset_feature_cols = [c for c in feature_cols if c.startswith(f"{asset}_")]
        assert len(asset_feature_cols) > 0, f"Should have features for {asset}"

    print(f"\n{'=' * 60}")
    print("TEST PASSED: ETF bundle computation")
    print(f"{'=' * 60}")


def test_feature_types_list():
    """
    Verify that the feature types list contains expected features.
    """
    expected_features = [
        FeatureType.PRICE,
        FeatureType.RETURN,
        FeatureType.LOG_RETURN,
        FeatureType.VOLUME,
        FeatureType.HIST_VOLATILITY,
        FeatureType.LAG_DELTAS,
        FeatureType.RSI,
        FeatureType.EMA,
        FeatureType.SPREAD_REL_EMA,
        FeatureType.DIFF_REL_EMA_MID,
        FeatureType.ADI,
        FeatureType.ROC,
    ]

    assert FEATURE_TYPES == expected_features, "Feature types should match expected list"
    print(f"Feature types verified: {len(FEATURE_TYPES)} features")


if __name__ == "__main__":
    test_compute_features_for_single_asset()
    test_compute_etf_bundle_small()
    test_feature_types_list()

Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/QQQ/QQQ_*_*_candle_1hour.df.parquet
  Loaded 92274 rows, 8 columns
  Result: 92274 rows, 119 features
  Saved: /app/data/bundle/candle_1hour/QQQ_features.parquet

Output file: /app/data/bundle/candle_1hour/QQQ_features.parquet
Shape: (92274, 123)
Columns: ['i_minute_i', 'S_open_f32', 'S_high_f32', 'S_low_f32', 'S_close_f32', 'S_volume_f64', 'S_open_time_i', 'S_close_time_i', 'minute_diff', 'gap_flag']...

TEST PASSED: Single ETF feature computation

Processing: VOO
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VOO/VOO_*_*_candle_1hour.df.parquet
  Loaded 47171 rows, 8 columns
  Result: 47171 rows, 119 features
  Saved: /app/data/bundle/candle_1hour/VOO_features.parquet

Processing: VEA
Loading: /app/data/normalised/candle_1hour/firstrate_undefined/etf/VEA/VEA_*_*_candle_1hour.df.parquet
  Loaded 41306 rows, 8 columns
  Result: 41306 rows, 119 features
  Saved: /app/data/bundle/candle_1hour/VEA_featu

In [19]:
test_file = '/app/data/bundle/test_assets_features_bundle.parquet'

df = pd.read_parquet(test_file)
print(f"Shape: {df.shape}")
print(f"Index: {df.index.name}")

feature_cols = [c for c in df.columns if "_F_" in c]
print(f"Feature columns: {len(feature_cols)}")
s_cols = [c for c in df.columns if "_S_" in c]
print(f"S columns: {len(s_cols)}")

Shape: (22281, 2441)
Index: None
Feature columns: 2220
S columns: 140


In [20]:
feature_cols
s_cols

['VOO_S_open_f32',
 'VOO_S_high_f32',
 'VOO_S_low_f32',
 'VOO_S_close_f32',
 'VOO_S_volume_f64',
 'VOO_S_open_time_i',
 'VOO_S_close_time_i',
 'VEA_S_open_f32',
 'VEA_S_high_f32',
 'VEA_S_low_f32',
 'VEA_S_close_f32',
 'VEA_S_volume_f64',
 'VEA_S_open_time_i',
 'VEA_S_close_time_i',
 'VWO_S_open_f32',
 'VWO_S_high_f32',
 'VWO_S_low_f32',
 'VWO_S_close_f32',
 'VWO_S_volume_f64',
 'VWO_S_open_time_i',
 'VWO_S_close_time_i',
 'IEF_S_open_f32',
 'IEF_S_high_f32',
 'IEF_S_low_f32',
 'IEF_S_close_f32',
 'IEF_S_volume_f64',
 'IEF_S_open_time_i',
 'IEF_S_close_time_i',
 'LQD_S_open_f32',
 'LQD_S_high_f32',
 'LQD_S_low_f32',
 'LQD_S_close_f32',
 'LQD_S_volume_f64',
 'LQD_S_open_time_i',
 'LQD_S_close_time_i',
 'VNQ_S_open_f32',
 'VNQ_S_high_f32',
 'VNQ_S_low_f32',
 'VNQ_S_close_f32',
 'VNQ_S_volume_f64',
 'VNQ_S_open_time_i',
 'VNQ_S_close_time_i',
 'GLD_S_open_f32',
 'GLD_S_high_f32',
 'GLD_S_low_f32',
 'GLD_S_close_f32',
 'GLD_S_volume_f64',
 'GLD_S_open_time_i',
 'GLD_S_close_time_i',
 'DBC_